# Vietnamese TTS: viXTTS — Voice Cloning for Vietnamese

## The Problem
None of the mainstream TTS models support Vietnamese well:
- **Parler-TTS**: English only
- **Bark**: Has Vietnamese presets but quality is poor
- **XTTS-v2**: 17 languages but **no Vietnamese**

## The Solution: viXTTS
[viXTTS](https://github.com/thinhlpg/TTS) is a **Vietnamese fork of XTTS** by thinhlpg that:
- Adds Vietnamese language support to XTTS architecture
- Uses `vinorm` for Vietnamese text normalization (numbers, abbreviations)
- Uses `underthesea` for Vietnamese sentence tokenization
- Supports **voice cloning** — clone any Vietnamese voice from 6-30s audio
- Pre-trained model on HuggingFace: `thinhlpg/viXTTS`

### Architecture (same as XTTS + Vietnamese patches)
```
Vietnamese Text ──▶ [vinorm] ──▶ Normalized Text
  "20/11"              │          "hai mươi tháng mười một"
                       ▼
Reference Audio ──▶ [Speaker Encoder] ──▶ Speaker Embedding
                                               │
Normalized Text ──▶ [GPT-2 Decoder]  ──▶ [HiFi-GAN] ──▶ 🔊 Audio
                     (vi-finetuned)
```

### Also available: vnTTS
Another approach by `anhnh2002/vnTTS` — similar XTTS fine-tune for Vietnamese.

---
**Kaggle Setup**: GPU T4 x2 | Internet enabled

## Step 1: Install Dependencies (~5 min)

In [ ]:
%%time
# Install Vietnamese XTTS fork (adds 'vi' language support)
!rm -rf TTS/
!git clone --branch add-vietnamese-xtts -q https://github.com/thinhlpg/TTS.git
!pip install --use-deprecated=legacy-resolver -q -e TTS
!pip install -q vinorm==2.0.7 cutlet underthesea unidic==1.1.0
!python -m unidic download 2>/dev/null
!pip install -q --force-reinstall 'protobuf>=5.26.1'

In [ ]:
# Download pre-trained viXTTS model from HuggingFace
import os
os.environ["COQUI_TOS_AGREED"] = "1"

from huggingface_hub import snapshot_download

print("Downloading viXTTS model (~1.8GB)...")
snapshot_download(
    repo_id="thinhlpg/viXTTS",
    repo_type="model",
    local_dir="model"
)
print("Model downloaded!")

# Check downloaded files
for f in os.listdir("model"):
    size = os.path.getsize(f"model/{f}") / 1e6
    print(f"  {f}: {size:.1f} MB")

## Step 2: Load Model

In [ ]:
import torch
import torchaudio
import time
from TTS.tts.configs.xtts_config import XttsConfig
from TTS.tts.models.xtts import Xtts
from IPython.display import Audio, display

device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

# Load viXTTS model
print("Loading viXTTS model...")
t0 = time.time()

config = XttsConfig()
config.load_json("model/config.json")

model = Xtts.init_from_config(config)
model.load_checkpoint(
    config,
    checkpoint_path="model/model.pth",
    vocab_path="model/vocab.json",
    use_deepspeed=False,  # set True if you installed deepspeed
)
model.to(device)

print(f"Model loaded in {time.time()-t0:.1f}s")

## Step 3: Vietnamese Text Normalization

Vietnamese TTS needs special text preprocessing:
- `20/11` → `hai mươi tháng mười một`
- `AI` → `Ây Ai`
- `100.000đ` → `một trăm nghìn đồng`
- Handle diacritics: `ă, â, ơ, ư, đ, ...`

This is handled by `vinorm` (Vietnamese Text Normalization).

In [ ]:
from vinorm import TTSnorm
from underthesea import sent_tokenize

# Demo: Vietnamese text normalization
examples = [
    "Hôm nay là ngày 20/11, nhiệt độ ngoài trời là 35°C.",
    "Công ty ABC có doanh thu 100.000.000đ trong Q3/2024.",
    "AI và Machine Learning đang thay đổi thế giới.",
    "Địa chỉ: 123 Nguyễn Huệ, Q.1, TP.HCM.",
]

print("Vietnamese Text Normalization Examples:")
print("=" * 60)
for text in examples:
    normalized = TTSnorm(text, unknown=False, lower=False, rule=True)
    print(f"\nOriginal:   {text}")
    print(f"Normalized: {normalized}")

## Step 4: Generate Vietnamese Speech

The model comes with pre-built Vietnamese voice samples:
- `nam-calm.wav` — Male, calm
- `nam-truyen-cam.wav` — Male, expressive
- `nu-calm.wav` — Female, calm
- `nu-nhe-nhang.wav` — Female, gentle
- `nu-luu-loat.wav` — Female, fluent

In [ ]:
def normalize_vietnamese(text):
    """Normalize Vietnamese text for TTS."""
    text = (
        TTSnorm(text, unknown=False, lower=False, rule=True)
        .replace("..", ".")
        .replace("!.", "!")
        .replace("?.", "?")
        .replace(" .", ".")
        .replace(" ,", ",")
        .replace('"', "")
        .replace("'", "")
        .replace("AI", "Ây Ai")
        .replace("A.I", "Ây Ai")
    )
    return text


def speak_vietnamese(model, text, speaker_wav, normalize=True, device="cuda:0"):
    """Generate Vietnamese speech with voice cloning."""
    # Get speaker embedding from reference audio
    gpt_cond_latent, speaker_embedding = model.get_conditioning_latents(
        audio_path=speaker_wav,
        gpt_cond_len=model.config.gpt_cond_len,
        max_ref_length=model.config.max_ref_len,
        sound_norm_refs=model.config.sound_norm_refs,
    )
    
    # Normalize text
    if normalize:
        text = normalize_vietnamese(text)
    
    # Split into sentences
    sentences = sent_tokenize(text)
    
    wav_chunks = []
    for sent in sentences:
        if not sent.strip():
            continue
        wav_chunk = model.inference(
            text=sent,
            language="vi",
            gpt_cond_latent=gpt_cond_latent,
            speaker_embedding=speaker_embedding,
            temperature=0.3,
            length_penalty=1.0,
            repetition_penalty=10.0,
            top_k=30,
            top_p=0.85,
        )
        wav_chunks.append(torch.tensor(wav_chunk["wav"]))
    
    out_wav = torch.cat(wav_chunks, dim=0).unsqueeze(0)
    return out_wav

print("Vietnamese TTS function ready!")

In [ ]:
# List available voice samples
import glob
samples = sorted(glob.glob("model/samples/*.wav"))
print("Available Vietnamese voice samples:")
for s in samples:
    print(f"  🎙️ {os.path.basename(s)}")
    display(Audio(s))

In [ ]:
# Generate Vietnamese speech!
text = "Xin chào! Tôi là trợ lý giọng nói tiếng Việt. Tôi có thể đọc bất kỳ văn bản nào bằng giọng của bạn."

# Try with female calm voice
speaker_wav = "model/samples/nu-calm.wav"
print(f"Speaker: {os.path.basename(speaker_wav)}")
print(f"Text: {text}")

t0 = time.time()
audio = speak_vietnamese(model, text, speaker_wav, device=device)
print(f"\nGenerated in {time.time()-t0:.1f}s")

display(Audio(audio, rate=24000))
torchaudio.save("vi_output_1.wav", audio, 24000)

## Step 5: Compare Different Vietnamese Voices

In [ ]:
text = "Trí tuệ nhân tạo đang thay đổi cách chúng ta sống và làm việc mỗi ngày."

voice_samples = {
    "Nam - Bình tĩnh": "model/samples/nam-calm.wav",
    "Nam - Truyền cảm": "model/samples/nam-truyen-cam.wav",
    "Nữ - Bình tĩnh": "model/samples/nu-calm.wav",
    "Nữ - Nhẹ nhàng": "model/samples/nu-nhe-nhang.wav",
}

for name, wav_path in voice_samples.items():
    if os.path.exists(wav_path):
        print(f"\n{'='*50}")
        print(f"🎙️ {name}")
        t0 = time.time()
        audio = speak_vietnamese(model, text, wav_path, device=device)
        print(f"Generated in {time.time()-t0:.1f}s")
        display(Audio(audio, rate=24000))
    else:
        print(f"\n⚠️ {name}: file not found at {wav_path}")

## Step 6: Long Text — Vietnamese Article Reading

In [ ]:
article = """
Việt Nam là một quốc gia nằm ở phía đông bán đảo Đông Dương thuộc khu vực Đông Nam Á.
Với dân số khoảng 100 triệu người, Việt Nam là quốc gia đông dân thứ 15 trên thế giới.
Thủ đô Hà Nội là trung tâm chính trị và văn hóa của cả nước.
Thành phố Hồ Chí Minh là thành phố lớn nhất và là trung tâm kinh tế quan trọng nhất.
""".strip()

print("Generating Vietnamese article reading...")
print(f"Text length: {len(article)} chars")

speaker_wav = "model/samples/nam-truyen-cam.wav"
t0 = time.time()
audio = speak_vietnamese(model, article, speaker_wav, device=device)
print(f"\nGenerated in {time.time()-t0:.1f}s")
print(f"Audio duration: {audio.shape[1]/24000:.1f}s")

display(Audio(audio, rate=24000))
torchaudio.save("vi_article.wav", audio, 24000)
print("Saved to vi_article.wav")

## Step 7: Voice Cloning — Use Your Own Voice

Upload a WAV file of your voice (6-30 seconds, clean recording) and the model will speak in your voice!

In [ ]:
# To clone your own voice:
# 1. Upload a WAV file to Kaggle (Settings → Data → Upload)
# 2. Update the path below

# my_voice = "/kaggle/input/my-voice/recording.wav"
# text = "Đây là giọng nói của tôi được nhân bản bởi trí tuệ nhân tạo."
# audio = speak_vietnamese(model, text, my_voice, device=device)
# display(Audio(audio, rate=24000))

print("To clone your voice:")
print("1. Record 10-30s of clear Vietnamese speech")
print("2. Save as WAV (22050 Hz, mono)")
print("3. Upload as Kaggle dataset")
print("4. Uncomment the code above and update the path")

## Key Takeaways

### Vietnamese TTS Options

| Model | Source | Voice Cloning | Quality | Notes |
|-------|--------|---------------|---------|-------|
| **viXTTS** | `thinhlpg/viXTTS` | Yes | Best | XTTS fork with Vietnamese |
| **vnTTS** | `anhnh2002/vnTTS` | Yes | Good | Another XTTS fine-tune |
| **Edge TTS** | Microsoft | No (fixed voices) | Good | Free API, no GPU needed |
| **gTTS** | Google | No | Basic | Very simple, no cloning |

### Key Vietnamese-specific components:
- **`vinorm`** — text normalization (numbers, dates, abbreviations)
- **`underthesea`** — sentence tokenization for Vietnamese
- **Voice samples** — male/female with different speaking styles

### Limitations:
- Based on XTTS architecture (can be slow on T4)
- Text normalization isn't perfect for all edge cases
- Model size ~1.8GB

### Next steps:
- Try with your own Vietnamese voice recordings
- Fine-tune further with more Vietnamese data
- Combine with Vietnamese STT (Whisper) for full voice pipeline